# 🎯 Week 3 — YOLOv8 Object Detection
**Dataset:** nikitamanaenkov/ultra-wide-fundus-images-for-tumor-diagnosis (Kaggle)
**Method:** GradCAM auto-labels → YOLOv8n training → mAP evaluation
**Classes:** CH, CO, Normal, RCH, RB, UM (6 classes)

In [ ]:
import os, json

# ── SET YOUR CREDENTIALS ────────────────────────────────────────────────────
KAGGLE_TOKEN = 'KGAT_13b2796488188aefc81cee8ba681cec6'  # your token
KAGGLE_USER  = 'YOUR_KAGGLE_USERNAME'                    # ← change this
# ────────────────────────────────────────────────────────────────────────────

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USER, 'key': KAGGLE_TOKEN}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✓ Kaggle credentials configured')

!pip install -q kaggle
print('⬇️  Downloading dataset...')
!kaggle datasets download -d nikitamanaenkov/ultra-wide-fundus-images-for-tumor-diagnosis --unzip -p /content/dataset

if os.path.exists('/content/dataset'):
    print('✓ Download successful')
    for root, dirs, files in os.walk('/content/dataset'):
        depth = root.replace('/content/dataset','').count('/')
        if depth > 3: continue
        n = len([f for f in files if f.endswith(('.jpg','.png'))])
        if n > 0:
            print(f'  {root}  →  {n} images')
else:
    print('❌ Download FAILED — check KAGGLE_USER and KAGGLE_TOKEN above')

In [ ]:
from google.colab import files
from pathlib import Path
import os, json

print('📁 Upload best_model.pth and classes.json ...')
uploaded = files.upload()

os.makedirs('/content/model', exist_ok=True)
for fname in uploaded.keys():
    src = f'/content/{fname}'
    dst = f'/content/model/{fname}'
    if os.path.exists(src):
        os.rename(src, dst)
    print(f'  ✓ {fname} → /content/model/')

# Auto-detect dataset root
def find_data_root(base):
    base = Path(base)
    for p in sorted(base.rglob('Training')):
        if p.is_dir(): return p.parent
    for p in sorted(base.rglob('*.jpg'))[:1]:
        return p.parent.parent
    return base

DATA_ROOT = find_data_root('/content/dataset')
TRAIN_DIR = DATA_ROOT / 'Training' if (DATA_ROOT / 'Training').exists() else DATA_ROOT
TEST_DIR  = DATA_ROOT / 'Testing'  if (DATA_ROOT / 'Testing').exists()  else DATA_ROOT
MODEL_PATH   = Path('/content/model/best_model.pth')
CLASSES_PATH = Path('/content/model/classes.json')

print(f'\n✓ TRAIN_DIR : {TRAIN_DIR}  exists={TRAIN_DIR.exists()}')
print(f'✓ TEST_DIR  : {TEST_DIR}   exists={TEST_DIR.exists()}')
print(f'✓ MODEL     : {MODEL_PATH.exists()}')
print('\nClass folders:')
for p in sorted(TEST_DIR.iterdir()):
    if p.is_dir():
        imgs = list(p.glob('*.jpg')) + list(p.glob('*.png'))
        print(f'  {p.name:<45} {len(imgs):>5} images')

In [ ]:
!pip install -q ultralytics torch torchvision opencv-python-headless Pillow numpy tqdm
from ultralytics import YOLO
import torch
print(f'✓ Ultralytics YOLOv8 installed')
print(f'✓ PyTorch {torch.__version__}')
print(f'✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import cv2
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

with open(CLASSES_PATH) as f:
    CLASS_NAMES = json.load(f)
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# Build classifier
def build_clf(n):
    m = models.efficientnet_b0(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n))
    return m

clf = build_clf(NUM_CLASSES)
clf.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
clf = clf.to(DEVICE).eval()
print(f'✓ Classifier loaded — {NUM_CLASSES} classes')

# GradCAM
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.grads = self.acts = None
        layer = model.features[-1]
        layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))
    def generate(self, tensor):
        self.model.zero_grad()
        out = self.model(tensor)
        cls_idx = out.argmax(1).item()
        out[0, cls_idx].backward()
        w = self.grads.mean(dim=(2,3), keepdim=True)
        cam = torch.relu((w * self.acts).sum(1)).squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls_idx

gradcam = GradCAM(clf)

clf_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def cam_to_yolo(cam, img_w, img_h, thr=0.40):
    cam_r = cv2.resize(cam, (img_w, img_h))
    binary = (cam_r >= thr).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.5, 0.5, 0.4, 0.4
    pts = np.concatenate(contours)
    x, y, w, h = cv2.boundingRect(pts)
    pad_x = int(w * 0.10); pad_y = int(h * 0.10)
    x = max(0, x-pad_x); y = max(0, y-pad_y)
    w = min(img_w-x, w+2*pad_x); h = min(img_h-y, h+2*pad_y)
    return round((x+w/2)/img_w,6), round((y+h/2)/img_h,6), round(w/img_w,6), round(h/img_h,6)

def annotate_split(split_dir, split_name, yolo_base):
    img_out = yolo_base / 'images' / split_name
    lbl_out = yolo_base / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    total = 0
    for cls_name in CLASS_NAMES:
        cls_dir = split_dir / cls_name
        if not cls_dir.exists(): continue
        cls_id = CLASS_TO_ID[cls_name]
        imgs = sorted(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
        for img_path in tqdm(imgs, desc=f'  {cls_name[:28]:<28}'):
            try:
                pil = Image.open(img_path).convert('RGB')
                ow, oh = pil.size
                tensor = clf_tf(pil).unsqueeze(0).to(DEVICE)
                cam, _ = gradcam.generate(tensor)
                cx, cy, nw, nh = cam_to_yolo(cam, ow, oh)
                shutil.copy2(img_path, img_out / img_path.name)
                with open(lbl_out / (img_path.stem + '.txt'), 'w') as f:
                    f.write(f'{cls_id} {cx} {cy} {nw} {nh}\n')
                total += 1
            except: pass
    return total

YOLO_DIR = Path('/content/yolo_dataset')
print('\n📦 Generating YOLO labels for Training set...')
n_train = annotate_split(TRAIN_DIR, 'train', YOLO_DIR)
print(f'\n📦 Generating YOLO labels for Testing set...')
n_val   = annotate_split(TEST_DIR,  'val',   YOLO_DIR)
print(f'\n✓ Train: {n_train} images labeled')
print(f'✓ Val  : {n_val} images labeled')

In [ ]:
yaml_content = f"""# YOLOv8 Dataset — Fundus Tumor Detection
path: /content/yolo_dataset
train: images/train
val:   images/val

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""
yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('✓ dataset.yaml created:')
print(yaml_content)

In [ ]:
from ultralytics import YOLO
import os

# Use YOLOv8n (nano) — fastest, good for Colab free tier
# Change to yolov8s.pt for better accuracy if you have more time
model_yolo = YOLO('yolov8n.pt')

print('🚀 Starting YOLOv8 training...')
print(f'   Dataset : {yaml_path}')
print(f'   Epochs  : 50')
print(f'   Device  : {DEVICE}')
print()

results = model_yolo.train(
    data    = str(yaml_path),
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = '/content/yolo_runs',
    name    = 'fundus_tumor',
    patience= 10,          # early stopping
    save    = True,
    plots   = True,
    verbose = True,
)

print('\n✓ Training complete!')
print(f'  Best model: {results.save_dir}/weights/best.pt')

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load best trained model
best_pt = Path('/content/yolo_runs/fundus_tumor/weights/best.pt')
model_eval = YOLO(str(best_pt))

print('📊 Running evaluation on test set...')
metrics = model_eval.val(
    data   = str(yaml_path),
    imgsz  = 640,
    device = 0 if torch.cuda.is_available() else 'cpu',
    plots  = True,
    save_json = True,
)

print('\n' + '='*55)
print('  YOLO DETECTION RESULTS')
print('='*55)
print(f'  mAP@0.5      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.2f}%)')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.2f}%)')
print(f'  Precision    : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.2f}%)')
print(f'  Recall       : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.2f}%)')
print('='*55)
print('\nPer-Class AP@0.5:')
for i, cls_name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f'  {cls_name:<42} {metrics.box.ap50[i]*100:.2f}%')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np
from PIL import Image

# Run predictions on test images (2 per class)
OUT_DIR = Path('/content/yolo_results')
OUT_DIR.mkdir(exist_ok=True)

sample_images = []
for cls_name in CLASS_NAMES:
    cls_dir = TEST_DIR / cls_name
    if not cls_dir.exists(): continue
    imgs = sorted(list(cls_dir.glob('*.jpg')))[:2]
    for img in imgs:
        sample_images.append((img, cls_name))

print(f'Running predictions on {len(sample_images)} sample images...')
results_list = []
for img_path, true_cls in sample_images:
    res = model_eval.predict(str(img_path), conf=0.25, verbose=False)
    results_list.append((img_path, true_cls, res[0]))

# Plot grid
n = min(12, len(results_list))
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4))
axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

CLASS_COLORS_HEX = {
    'Choroidal Hemangioma (CH)':'#FF6B6B','Choroidal Osteoma (CO)':'#4ECDC4',
    'Normal':'#95E1D3','Retinal Capillary Hemangioma (RCH)':'#FFD93D',
    'Retinoblastoma (RB)':'#FF006E','Uveal Melanoma (UM)':'#6A4C93'
}

for i, (img_path, true_cls, res) in enumerate(results_list[:n]):
    ax = axes[i]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    # Draw predicted boxes
    if res.boxes is not None and len(res.boxes) > 0:
        for box in res.boxes:
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            pred_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else 'Unknown'
            color = CLASS_COLORS_HEX.get(pred_name, '#FF0000')
            rect = patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2.5,edgecolor=color,facecolor='none')
            ax.add_patch(rect)
            short = pred_name.split('(')[-1].replace(')','').strip()
            ax.text(x1, max(y1-5,10), f'{short} {conf:.2f}', color=color,
                    fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))
    short_true = true_cls.split('(')[-1].replace(')','').strip()
    ax.set_title(f'GT: {short_true}', fontsize=9, fontweight='bold')
    ax.axis('off')

for j in range(n, len(axes)): axes[j].axis('off')
plt.suptitle('YOLOv8 Detection Results — Fundus Tumor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'yolo_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: yolo_predictions.png')

In [ ]:
import shutil, json
from pathlib import Path

RUNS_DIR = Path('/content/yolo_runs/fundus_tumor')

# Copy training plots
plots_to_copy = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'PR_curve.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'val_batch0_pred.jpg',
]
for plot in plots_to_copy:
    src = RUNS_DIR / plot
    if src.exists():
        shutil.copy2(src, OUT_DIR / plot)
        print(f'✓ Copied: {plot}')

# Save metrics JSON
metrics_data = {
    'mAP50':      round(float(metrics.box.map50), 4),
    'mAP50_95':   round(float(metrics.box.map),   4),
    'precision':  round(float(metrics.box.mp),    4),
    'recall':     round(float(metrics.box.mr),    4),
    'per_class_ap50': {
        CLASS_NAMES[i]: round(float(metrics.box.ap50[i]), 4)
        for i in range(min(len(CLASS_NAMES), len(metrics.box.ap50)))
    },
    'model': 'YOLOv8n',
    'epochs': 50,
    'img_size': 640,
    'labels': 'GradCAM auto-generated',
}
with open(OUT_DIR / 'yolo_metrics.json', 'w') as f:
    json.dump(metrics_data, f, indent=2)
print('✓ Saved: yolo_metrics.json')

print('\n' + '='*55)
print('  FINAL RESULTS SUMMARY')
print('='*55)
print(f'  mAP@0.5      : {metrics_data["mAP50"]*100:.2f}%')
print(f'  mAP@0.5:0.95 : {metrics_data["mAP50_95"]*100:.2f}%')
print(f'  Precision    : {metrics_data["precision"]*100:.2f}%')
print(f'  Recall       : {metrics_data["recall"]*100:.2f}%')
print('='*55)

In [ ]:
import zipfile
from google.colab import files
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name  = f'/content/Week3_YOLO_Results_{timestamp}.zip'

# Collect all result files
result_files = list(OUT_DIR.glob('*')) + [best_pt]

print('📦 Creating ZIP...')
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in result_files:
        if Path(f).exists():
            zf.write(f, Path(f).name)
            print(f'  ✓ {Path(f).name}  ({Path(f).stat().st_size/1024:.1f} KB)')

zip_size = Path(zip_name).stat().st_size / (1024*1024)
print(f'\n✓ ZIP: {Path(zip_name).name}  ({zip_size:.2f} MB)')
print('⬇️  Downloading...')
files.download(zip_name)
print('✅ Done!')